# Prédiction de la valeur marchande d'un joueur

Ce notebook permet d'estimer la valeur marchande d'un joueur de football à partir du modèle final
entraîné dans le notebook `09_modelisation_finale.ipynb`.

**Il est autosuffisant** : il ne dépend que d'un seul fichier, `../modelisation/pipeline_explicabilite.pkl`
(généré à la fin du notebook 9), qui contient les modèles déjà entraînés. Pas besoin de ré-entraîner quoi
que ce soit ni de relancer les autres notebooks.

## Mode d'emploi rapide

1. Exécuter toutes les cellules des sections **1** et **2** (chargement + modèle).
2. Prédictions au cas par cas :
   - **Section 3** : rechercher un joueur déjà présent dans les données, par son nom.
   - **Section 4** : simuler un joueur en saisissant seulement les caractéristiques qu'on connaît.
3. Exports pour PowerBI :
   - **Section 5** : historique des VM réelles vs prédites, pour **tous les joueurs**.
   - **Section 6** : table de sensibilité pour des curseurs interactifs (onglet "Prédiction / simulateur").

## 1. Chargement du modèle et des données

On recharge tout ce qui est nécessaire depuis le pipeline sauvegardé par le notebook 9. Rien d'autre
n'est requis (pas de ré-entraînement, pas de fichier de configuration supplémentaire).

In [68]:
import numpy as np
import pandas as pd
import difflib
import os
import sys
from datetime import datetime

import joblib
from sklearn.base import BaseEstimator, RegressorMixin

# On connecte le notebook à tous les fichiers inclus dans le dossier /fonctions
sys.path.insert(0, os.path.abspath("../fonctions/modelisations"))

%load_ext autoreload
%autoreload 2

# On importe les fonctions utiles à la modélisation
from prediction_joueur import *

pipeline = joblib.load("../modelisation/pipeline_explicabilite.pkl")

X_train = pipeline["X_train"]
X_val = pipeline["X_val"]
X_test = pipeline["X_test"]
X_en_cours = pipeline["X_en_cours"]

df_train = pipeline["df_train"]
df_val = pipeline["df_val"]
df_test = pipeline["df_test"]
df_en_cours = pipeline["df_en_cours"]

modeles_finaux = pipeline["modeles_finaux"]   # XGBoost / LightGBM / CatBoost déjà entraînés
meta = pipeline["meta"]                        # méta-modèle du stacking, déjà entraîné
colonne_cible = pipeline["colonne_cible"]       # "market_value_in_eur"

colonne_joueur = "player"
colonne_saison = "season_year"
colonne_team = "team"

print("Pipeline chargé avec succès.")
print(f"{X_train.shape[1]} variables attendues par le modèle.")

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
Pipeline chargé avec succès.
146 variables attendues par le modèle.


## 2. Le modèle de prédiction (Stacking)

Le modèle final combine 3 modèles (XGBoost, LightGBM, CatBoost) : chacun propose une estimation, puis un méta-modèle apprend la meilleure combinaison des trois pour donner la prédiction finale, directement en euros.

In [69]:
modele_stack = StackingModel(
    models=[
        modeles_finaux["XGBoost (log)"],
        modeles_finaux["LightGBM (log)"],
        modeles_finaux["CatBoost (log)"],
    ],
    meta_model=meta,
)

print("Modèle de stacking prêt.")

Modèle de stacking prêt.


## 3. Mode 1 — Rechercher un joueur existant

Recherche un joueur par son nom parmi toutes lesnsaisons disponibles (train / validation / test / en cours). Par défaut, la saison la plus récente disponible est utilisée.

In [70]:
# Historique de toutes les prédictions réalisées dans cette session
historique_predictions = []

DATASETS_PAR_SPLIT = {
    "train": (df_train, X_train),
    "val": (df_val, X_val),
    "test": (df_test, X_test),
    "en_cours": (df_en_cours, X_en_cours),
}

**Exemple d'utilisation** — à remplacer par le nom de votre choix :

In [71]:
# Exemple : remplacer le nom par celui d'un joueur présent dans vos données
predire_joueur_existant("Kylian Mbappé", DATASETS_PAR_SPLIT, modele_stack, historique_predictions)

# Pour une saison précise (au lieu de la plus récente) :
predire_joueur_existant("Kylian Mbappé", DATASETS_PAR_SPLIT, modele_stack, historique_predictions, saison=2022)

ℹ️ 6 saisons trouvées pour Kylian Mbappé, on utilise la plus récente (2025). Précisez saison=... pour en choisir une autre.
👤 Kylian Mbappé
   Saison               : 2025
   💰 Valeur marchande prédite : 110 650 802 €
   📊 Valeur réelle connue    : 200 000 000 €
   📈 Écart                   : -44.7 %
👤 Kylian Mbappé
   Saison               : 2022
   💰 Valeur marchande prédite : 159 285 884 €
   📊 Valeur réelle connue    : 180 000 000 €
   📈 Écart                   : -11.5 %


np.float64(159285884.31058916)

## 4. Mode 2 — Saisir un joueur manuellement

Pour simuler un joueur (nouveau, hypothétique, ou dont on ne connaît que quelques caractéristiques), il n'est **pas nécessaire de renseigner les ~150 variables du modèle**. On part d'un profil "joueur moyen" (médiane des variables numériques, valeur la plus fréquente pour les variables 0/1, calculées automatiquement sur les données d'entraînement), et on ne renseigne que ce que l'on sait.

### Champs disponibles pour la saisie

| Clé à utiliser dans le dictionnaire | Description | Exemple de valeur |
|---|---|---|
| `age` | Âge du joueur | `24` |
| `taille_cm` | Taille en cm | `182` |
| `pied` | `"Droit"`, `"Gauche"` ou `"Ambidextre"` | `"Gauche"` |
| `poste` | `"Gardien"`, `"Défenseur"`, `"Milieu"` ou `"Attaquant"` | `"Attaquant"` |
| `championnat` | `"Premier League"`, `"La Liga"`, `"Ligue 1"`, `"Bundesliga"`, `"Serie A"` ou `"Autre / non renseigné"` | `"Ligue 1"` |
| `valeur_marchande_precedente_euros` | Valeur marchande de la saison précédente (facteur le plus important du modèle) | `18_000_000` |
| `classement_equipe` | Classement de l'équipe en championnat | `5` |
| `buts` | Buts marqués dans la saison | `12` |
| `passes_decisives` | Passes décisives dans la saison | `6` |
| `matchs_joues` | Nombre de matchs disputés | `30` |
| `minutes_jouees_par_match` | Minutes jouées par match disputé | `75` |

Tout champ non renseigné garde sa valeur par défaut (profil moyen).

**Exemple d'utilisation** — à adapter avec les caractéristiques du joueur simulé :

In [72]:
saisie_exemple = {
    "age": 20,
    "taille_cm": 180,
    "pied": "Gauche",
    "poste": "Attaquant",
    "championnat": "Ligue 1",
    "valeur_marchande_precedente_euros": 12_000_000,
    "classement_equipe": 4,
    "buts": 15,
    "passes_decisives": 5,
    "matchs_joues": 32,
    "minutes_jouees_par_match": 78,
}

predire_joueur_manuel(modele_stack, historique_predictions, saisie_exemple, X_train, nom_affiche="Exemple - jeune attaquant prometteur")

👤 Exemple - jeune attaquant prometteur
   💰 Valeur marchande prédite : 11 706 695 €


np.float64(11706695.387087906)

## 5. Export des prédictions de cette session

Toutes les prédictions réalisées ci-dessus (recherche ou saisie manuelle) sont exportées ici.

In [73]:
os.makedirs("../exports", exist_ok=True)

df_export = pd.DataFrame(historique_predictions)
chemin_csv = "../exports/predictions_session_powerbi.csv"

df_export.to_csv(chemin_csv, index=False, encoding="utf-8-sig", decimal=",")

print(f"{len(df_export)} prédiction(s) de cette session exportée(s) vers {chemin_csv}")
df_export

3 prédiction(s) de cette session exportée(s) vers ../exports/predictions_session_powerbi.csv


,date_prediction,joueur,mode,saison,valeur_predite_euros,valeur_reelle_euros,ecart_pct
0,2026-09-09 10:26,Kylian Mbappé,recherche,2025.0,1.106508e+08,200000000.0,-44.67
1,2026-09-09 10:26,Kylian Mbappé,recherche,2022.0,1.592859e+08,180000000.0,-11.51
2,2026-09-09 10:26,Exemple - jeune attaquant prometteur,manuel,NaN,1.170670e+07,NaN,NaN


## 6. Export complet de tous les joueurs

Pour visualiser dans PowerBI l'évolution des VM de **tous** les joueurs (pas seulement ceux testés manuellement plus haut), on calcule ici la VM réelle **et** la VM prédite par le modèle pour chaque ligne joueur/saison disponible (train + validation + test + en cours). Cela permet, dans PowerBI, de comparer visuellement valeur réelle et valeur estimée par le modèle, saison après saison.

### Utilisation dans PowerBI
**Accueil → Obtenir les données → Texte/CSV**, sélectionner `historique_vm_joueurs_powerbi.csv`.

In [74]:
df_historique = construire_export_historique(datasets = DATASETS_PAR_SPLIT, modele_stack = modele_stack)

chemin_historique = "../exports/historique_vm_joueurs_powerbi.csv"
df_historique.to_csv(chemin_historique, index=False, encoding="utf-8-sig", decimal=",")

print(f"{len(df_historique)} lignes joueur/saison exportées vers {chemin_historique}")
df_historique.head()

14413 lignes joueur/saison exportées vers ../exports/historique_vm_joueurs_powerbi.csv


,joueur,equipe,saison,jeu,vm_reelle_euros,vm_predite_euros,ecart_pct
0,Aaron Connolly,Brighton,2020,train,7000000.0,9.076215e+06,0.30
1,Aaron Cresswell,West Ham United,2020,train,5000000.0,8.445674e+06,0.69
2,Aaron Cresswell,West Ham United,2021,train,3000000.0,4.345411e+06,0.45
3,Aaron Cresswell,West Ham United,2022,train,1200000.0,1.870255e+06,0.56
4,Aaron Hickey,Bologna,2020,train,5000000.0,5.216640e+06,0.04


## 7. Table de sensibilité pour les curseurs

PowerBI, sans Python configuré, ne peut pas appeler le modèle à chaque mouvement de curseur : un "What-if parameter" PowerBI ne fait que piloter une mesure DAX. Pour chaque variable qu'on veut rendre interactive, on précalcule la prédiction pour **toute une plage de valeurs réalistes**, les autres variables du joueur restant fixes. Le curseur PowerBI n'aura plus qu'à aller chercher la bonne ligne dans cette table, sans aucun calcul Python côté PowerBI.

4 curseurs sont préparés : **âge**, **buts**, **VM saison précédente**, **classement de l'équipe**.

Deux profils de référence sont exportés :
- **"Profil moyen"** : un joueur "type" (médiane des données d'entraînement).
- **un joueur réel** de votre choix.

### Utilisation dans PowerBI
1. Créer 4 **What-if parameters** (Modélisation → Nouveau paramètre → Champ numérique) : `Âge`, `Buts`,
   `VM saison précédente`, `Classement équipe`, avec les mêmes min/max/incrément que les plages ci-dessous.
2. Créer une mesure DAX par curseur, par exemple pour l'âge :
   ```
   VM prédite (curseur Âge) =
   LOOKUPVALUE(
       table_sensibilite[vm_predite_euros],
       table_sensibilite[variable], "age",
       table_sensibilite[valeur], [Âge (curseur)],
       table_sensibilite[profil_reference], SELECTEDVALUE(table_sensibilite[profil_reference], "Profil moyen")
   )
   ```
3. Ajouter une carte (visuel "Carte") affichant cette mesure : elle se met à jour dès que le curseur bouge.

In [ ]:
# Joueur moyen
ligne_moyenne = ligne_par_defaut(X_train)
table_profil_moyen = construire_table_sensibilite(modele_stack, X_train, ligne_moyenne, nom_reference = "Profil moyen")

# Joueur réel
nom_joueur_reference = "Kylian Mbappé"
correspondances = chercher_joueur(nom_joueur_reference, DATASETS_PAR_SPLIT)

if correspondances:
    nom_exact, _, split, idx = max(correspondances, key=lambda c: c[1])
    _, X_split_ref = DATASETS_PAR_SPLIT[split]
    ligne_joueur_reel = X_split_ref.loc[[idx]].reset_index(drop=True)
    table_profil_joueur = construire_table_sensibilite(modele_stack, X_train, ligne_joueur_reel, nom_reference = nom_exact)
    table_sensibilite = pd.concat([table_profil_moyen, table_profil_joueur], ignore_index=True)
else:
    print(f"\u26A0\uFE0F « {nom_joueur_reference} » non trouvé : seul le profil moyen est exporté.")
    table_sensibilite = table_profil_moyen

# On enregistre en format CSV
chemin_sensibilite = "../exports/table_sensibilite_powerbi.csv"
table_sensibilite.to_csv(chemin_sensibilite, index=False, encoding="utf-8-sig")

print(f"{len(table_sensibilite)} lignes exportées vers {chemin_sensibilite}")
print("Profils disponibles :", table_sensibilite["profil_reference"].unique().tolist())
table_sensibilite.head()

324 lignes exportées vers ../exports/table_sensibilite_powerbi.csv
Profils disponibles : ['Profil moyen', 'Kylian Mbappé']


,profil_reference,variable,valeur,vm_predite_euros
0,Profil moyen,age,15.0,1.274767e+07
1,Profil moyen,age,16.0,9.009946e+06
2,Profil moyen,age,17.0,7.847295e+06
3,Profil moyen,age,18.0,7.289722e+06
4,Profil moyen,age,19.0,6.943071e+06
